# Bloque B en Kaggle (lanzador): detección de anomalías en RAW-FABRID

Este notebook **no tiene lógica propia**: clona el repo, baja RAW-FABRID de Mendeley (4,1 GB, se verifica el SHA-256),
extrae sólo lo que usa el bloque B y ejecuta `notebooks/03_anomalias_raw_fabrid.ipynb`.

**Settings (panel derecho):** *Accelerator* = **None** (el bloque B corre en CPU; así no se gastan horas de GPU)
· *Internet* = **On**.

1. Probar con `MODO_PRUEBA = True` y **Run All** (la descarga tarda unos minutos).
2. Corrida real: `MODO_PRUEBA = False` y **Save Version → Save & Run All (Commit)**. Se puede cerrar la compu.
3. Al terminar, en *Output*: `results_bloque_b.zip` (descomprimir en la raíz del repo) y el notebook 03 ejecutado.

Si una corrida se corta, agregar su Output con **Add Input** y volver a correr: se reutiliza lo ya calculado.

In [ ]:
# ===== Parámetros: lo único que hay que tocar =====
MODO_PRUEBA = True       # True: corrida corta para verificar que todo anda. False: corrida real
RAMA = "main"            # rama del repo a usar

In [ ]:
NOTEBOOK = "03_anomalias_raw_fabrid.ipynb"
PARAMETROS = {"MODO_PRUEBA": MODO_PRUEBA}
ETIQUETA = "raw_fabrid"
ZIP_RESULTADOS = "results_bloque_b"
PAQUETES = "papermill"   # torch, torchvision, scikit-learn, scipy y opencv ya vienen en Kaggle

import os, subprocess, sys, shutil, glob, json
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "tp_final_vision"
DATA = Path("/tmp/rewear_data")   # fuera de /kaggle/working: los datos no ocupan lugar en el Output
os.environ["REWEAR_DATA_DIR"] = str(DATA)

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

if not REPO.exists():
    sh(f"git clone --depth 1 -b {RAMA} https://github.com/victoriapersico/tp_final_vision.git {REPO}")
else:
    sh(f"git -C {REPO} pull")
sh(f"git -C {REPO} log --oneline -1")
sh(f"{sys.executable} -m pip install -q {PAQUETES}")

sh("nproc && free -g | head -2")

In [ ]:
# Si se agregó como Input el Output de una versión anterior de este notebook, se recuperan sus resultados
# para no repetir lo que ya se calculó (los notebooks reutilizan todo lo que encuentran en results/).
for prev in glob.glob("/kaggle/input/**/tp_final_vision/results", recursive=True):
    print("Recuperando resultados previos de", prev)
    shutil.copytree(prev, REPO / "results", dirs_exist_ok=True)
for z in glob.glob(f"/kaggle/input/**/{ZIP_RESULTADOS}.zip", recursive=True):
    print("Recuperando resultados previos de", z)
    shutil.unpack_archive(z, REPO)

In [ ]:
# RAW-FABRID v1 desde la API pública de Mendeley (https://data.mendeley.com/datasets/db6g85xsyg/1).
import hashlib, urllib.request, zipfile

SHA256 = "f6b4f82cce2f0220bbab3968a74b5337ebf2bbfd5e33d0548d34e1ebf80f85b9"   # verificado en docs/DATASETS.md
destino = DATA / "raw_fabrid" / "extracted"
raw = destino / "RAW_FABRID"

if not (raw / "RAW_FABRID_Patches_Metadata.csv").exists():
    api = "https://data.mendeley.com/public-api/datasets/db6g85xsyg/files?folder_id=root&version=1"
    # Mendeley responde 403 al User-Agent por defecto de urllib, por eso se manda uno explícito.
    pedido = urllib.request.Request(api, headers={"User-Agent": "Wget/1.21"})
    archivo = json.load(urllib.request.urlopen(pedido))[0]
    url = archivo["content_details"]["download_url"]
    assert archivo["content_details"]["sha256_hash"] == SHA256, "Mendeley cambió el archivo: revisar la versión"
    zip_path = Path("/tmp/RAW_FABRID.zip")
    print(f"Descargando {archivo['filename']} ({archivo['size'] / 1e9:.1f} GB)...", flush=True)
    sh(f"wget -q --show-progress --progress=dot:giga -O {zip_path} '{url}'")
    h = hashlib.sha256()
    with zip_path.open("rb") as f:
        for bloque in iter(lambda: f.read(1 << 24), b""):
            h.update(bloque)
    assert h.hexdigest() == SHA256, "El SHA-256 no coincide: la descarga está corrupta"
    print("SHA-256 verificado.")

    # Sólo se extraen los patches MVTec y los metadatos de la raíz (el bloque B no usa las imágenes 1792x1024).
    destino.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        nombres = z.namelist()
        meta = next(n for n in nombres if n.endswith("RAW_FABRID_Patches_Metadata.csv"))
        prefijo = meta[: -len("RAW_FABRID_Patches_Metadata.csv")]
        elegidos = [n for n in nombres if n.startswith(prefijo) and
                    ("/MVTec/" in "/" + n[len(prefijo):] or "/" not in n[len(prefijo):])]
        print(f"Extrayendo {len(elegidos)} de {len(nombres)} archivos...", flush=True)
        z.extractall(destino, members=elegidos)
    zip_path.unlink()
    raiz = destino / prefijo
    if raiz.resolve() != raw.resolve():
        raw.parent.mkdir(parents=True, exist_ok=True)
        raw.symlink_to(raiz.resolve(), target_is_directory=True)

print("RAW-FABRID listo en", raw)
for sub in ("train/good", "test/good", "test/defect", "ground_truth/defect"):
    print(f"  MVTec/{sub}: {len(list((raw / 'MVTec' / sub).glob('*')))} archivos")

In [ ]:
# Copia del notebook con los parámetros fijados y ejecución de punta a punta.
src = REPO / "notebooks" / NOTEBOOK
nb = json.loads(src.read_text(encoding="utf-8"))
celda = next(c for c in nb["cells"] if c["cell_type"] == "code" and "MODO_PRUEBA =" in "".join(c["source"]))
lineas = "".join(celda["source"]).splitlines(keepends=True)
for i, linea in enumerate(lineas):
    for nombre, valor in PARAMETROS.items():
        if linea.startswith(f"{nombre} ="):
            lineas[i] = f"{nombre} = {valor!r}  # fijado por el lanzador de Kaggle" + chr(10)
celda["source"] = "".join(lineas)
print("".join(l for l in lineas if "lanzador de Kaggle" in l))

entrada = REPO / "notebooks" / f"_kaggle_{NOTEBOOK}"
entrada.write_text(json.dumps(nb, ensure_ascii=False), encoding="utf-8")
salida = WORK / f"{Path(NOTEBOOK).stem}_ejecutado_{ETIQUETA}{'_prueba' if MODO_PRUEBA else ''}.ipynb"

import papermill as pm
try:
    pm.execute_notebook(str(entrada), str(salida), cwd=str(REPO / "notebooks"), log_output=True, kernel_name="python3")
finally:
    # Aunque falle a mitad de camino, se empaqueta lo que se haya llegado a calcular.
    shutil.make_archive(str(WORK / ZIP_RESULTADOS), "zip", REPO, "results")
    print("Resultados en", WORK / f"{ZIP_RESULTADOS}.zip")
    print("Notebook ejecutado en", salida)